# 01 · Create Synthetic Clinical Profiles

This notebook turns each admission in `unified_table.csv` into structured, de-identified
**clinical profiles** that later feed the LLM note-writer (notebooks `02`/`03`). Everything
here is **programmatic** (rule/template based, no LLM) for full control and reproducibility.

**Pipeline (Stages A–D)**

1. **Stage A — factual profiles.** Each admission becomes several structured factual
   profiles (demographics, vitals, labs, procedures, microbiology), each written under a
   different `profile_focus` angle for diversity.
2. **Stage B — validate & checkpoint.** Sanity-check the always-present admission fields
   (age, gender, length of stay), drop failures, and save `clean_factual_profiles.jsonl`.
3. **Stage C — symptom enrichment.** `sample_symptoms(label, infection_source)` draws
   category-appropriate symptoms whose count is driven by a sampled *severity*, so the
   symptom picture stays clinically coherent with the true infection category.
4. **Stage D — validate & checkpoint.** Check severity/symptom consistency, scan for
   diagnostic leakage terms, and save `clean_symptom_profiles.jsonl`.

**Anti-leakage integrity** — profiles never contain antibiotic names, an unmasked infection
diagnosis, or the ground-truth infection label/category. Symptom strings are lay cues only
(never a diagnosis, organism, or the label), and a final scan asserts zero leakage terms.

**Outputs** (both under `data/01_profiles/`): `clean_factual_profiles.jsonl` and
`clean_symptom_profiles.jsonl`. The latter embeds the full factual profile, so it is the
single pre-LLM checkpoint consumed downstream.

## 1 · Setup

In [1]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Reproducibility: a single seeded RNG drives every random choice below.
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

# --- Paths (works from the repo root or from notebooks/) -------------------
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
PROFILES_DIR = DATA_DIR / "01_profiles"
PROFILES_DIR.mkdir(parents=True, exist_ok=True)  # ensure the output folder exists

UNIFIED_TABLE_PATH = DATA_DIR / "00_mimic_data/unified_table.csv"

print("Unified table :", UNIFIED_TABLE_PATH)
print("Profiles dir  :", PROFILES_DIR)

Unified table : C:\Users\talme\Documents\LLM PROJECT - Improved\data\00_mimic_data\unified_table.csv
Profiles dir  : C:\Users\talme\Documents\LLM PROJECT - Improved\data\01_profiles


C:\Users\talme\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 · Load the unified table

In [2]:
if not UNIFIED_TABLE_PATH.exists():
    raise FileNotFoundError(f"Could not find unified table: {UNIFIED_TABLE_PATH}")

df = pd.read_csv(UNIFIED_TABLE_PATH)

df["infection_label"] = pd.to_numeric(df["infection_label"], errors="coerce")
df["infection_label"] = df["infection_label"].astype(int)

print("Unified table shape:", df.shape)
print("Label distribution:\n", df["infection_label"].value_counts())
df.head()

Unified table shape: (275, 26)
Label distribution:
 infection_label
0    145
1    130
Name: count, dtype: int64


,subject_id,hadm_id,age,gender,admission_type,admission_location,los_days,hospital_expire_flag,Temp_max,HeartRate_max,...,chronic_conditions,procedures,antibiotics_given,organisms,specimen_types,any_hospital_onset_culture,infection_label,infection_categories,infection_diagnoses,num_infection_dx
0,10004235,24181354,47,M,URGENT,TRANSFER FROM HOSPITAL,8.98,0,38.5,144.0,...,"Atrial fibrillation, Acute systolic heart fail...","Arterial catheterization, Endoscopic insertion...",Empiric intravenous antibiotic therapy adminis...,NaN,BLOOD CULTURE; MRSA SCREEN; SPUTUM; URINE,False,1,Other/Mixed,"Pneumonia, organism unspecified; Cholangitis; ...",4
1,10009628,25926192,58,M,URGENT,TRANSFER FROM HOSPITAL,7.84,0,37.4,126.0,...,"Other and unspecified hyperlipidemia, Atrial f...",Single internal mammary-coronary artery bypass...,Empiric intravenous antibiotic therapy adminis...,NaN,MRSA SCREEN; Staph aureus swab; URINE,False,0,NaN,NaN,0
2,10018081,23983182,79,M,URGENT,TRANSFER FROM HOSPITAL,5.73,0,NaN,NaN,...,"Other and unspecified hyperlipidemia, Chronic ...","Percutaneous abdominal drainage, Parenteral in...",Empiric intravenous antibiotic therapy adminis...,CLOSTRIDIUM DIFFICILE,BLOOD CULTURE; STOOL,False,1,Skin/Soft tissue,Other postoperative infection; Peritoneal abscess,2
3,10006053,22942076,52,M,URGENT,TRANSFER FROM HOSPITAL,1.74,1,37.5,120.0,...,Alcoholic cirrhosis of liver,"Venous catheterization, not elsewhere classifi...",Empiric intravenous antibiotic therapy adminis...,NaN,BLOOD CULTURE; MRSA SCREEN; PERITONEAL FLUID; ...,False,0,NaN,NaN,0
4,10031404,21606243,82,F,URGENT,TRANSFER FROM HOSPITAL,2.09,0,37.1,134.0,...,"Essential (primary) hypertension, Hyperlipidem...","Drainage of Pericardial Cavity, Percutaneous A...",NaN,NaN,NaN,False,0,NaN,NaN,0


### Helper Functions

In [3]:
MISSING_VALUES = {"", "none", "nan", "null", "na", "n/a", "unknown", "not recorded"}

def is_missing_value(value):
    """Return True if a value should be treated as missing."""
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except TypeError:
        pass

    if isinstance(value, str) and value.strip().lower() in MISSING_VALUES:
        return True

    return False


def get_first_existing(row, possible_columns):
    """Return the first non-missing value from a list of possible columns."""
    for col in possible_columns:
        if col in row.index and not is_missing_value(row[col]):
            return row[col]
    return None


def clean_text_value(value):
    """Clean a text value without changing its meaning."""
    if is_missing_value(value):
        return None

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)

    return value


def clean_drug_name(value):
    """Normalize drug name casing to make generated notes look less artificial."""
    if is_missing_value(value):
        return None

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    # Keep common abbreviations readable.
    value = value.title()
    value = value.replace(" Iv", " IV")
    value = value.replace(" Po/Ng", " PO/NG")

    return value


def clean_administrative_value(value):
    """Convert administrative codes into more natural wording."""
    if is_missing_value(value):
        return None

    value = str(value).strip()
    value_upper = value.upper()
    replacements = {
        "URGENT": "urgent admission",
        "ELECTIVE": "elective admission",
        "EW EMER": "emergency admission",
        "OBSERVATION ADMIT": "observation admission",
        "DIRECT OBSERVATION": "direct observation",
        "TRANSFER FROM HOSPITAL": "transferred from another hospital",
        "TRANSFER FROM SKILLED NURSING FACILITY": "transferred from skilled nursing facility",
        "PHYSICIAN REFERRAL": "physician referral",
        "WALK-IN/SELF REFERRAL": "walk-in or self-referral"
    }

    return replacements.get(value_upper, value.lower())


def parse_semicolon_list(value, clean_function=None):
    """Parse semicolon/comma-separated values into a clean list."""
    if is_missing_value(value):
        return []
    value = str(value).strip()

    if ";" in value:
        items = [item.strip() for item in value.split(";")]
    elif "," in value:
        items = [item.strip() for item in value.split(",")]
    else:
        items = [value]

    cleaned_items = []
    for item in items:
        if not item or item.lower() in MISSING_VALUES:
            continue
        if clean_function is not None:
            item = clean_function(item)
        else:
            item = clean_text_value(item)

        if item:
            cleaned_items.append(item)

    return list(dict.fromkeys(cleaned_items))


def to_float_or_none(value):
    """Convert a value to float, or return None."""
    if is_missing_value(value):
        return None

    try:
        return float(value)
    except Exception:
        return None


def to_int_or_none(value):
    """Convert a value to int, or return None."""
    if is_missing_value(value):
        return None

    try:
        return int(float(value))
    except Exception:
        return None


def normalize_bool(value):
    """Normalize boolean-like values into True/False/None."""
    if is_missing_value(value):
        return None

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return bool(int(value))

    value_str = str(value).strip().lower()

    if value_str in {"1", "true", "yes"}:
        return True

    if value_str in {"0", "false", "no"}:
        return False

    return None


def map_label(infection_label):
    """Map numeric infection label to textual target label."""
    infection_label = int(infection_label)

    if infection_label == 1:
        return "High Infection Concern"

    if infection_label == 0:
        return "Low Infection Concern"

    raise ValueError(f"Invalid infection_label: {infection_label}")


def map_infection_source(row):
    """Map infection categories into internal metadata."""
    infection_label = int(row["infection_label"])

    if infection_label == 0:
        return "none"

    if "infection_categories" in row.index and not is_missing_value(row["infection_categories"]):
        return str(row["infection_categories"]).strip()

    return "documented_infection"

### Validate required columns

In [4]:
# Validate required columns for Stage A.
required_columns = [
    "subject_id",
    "hadm_id",
    "infection_label"
]

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

df["infection_label"] = pd.to_numeric(df["infection_label"], errors="coerce")
df = df.dropna(subset=["infection_label"]).copy()
df["infection_label"] = df["infection_label"].astype(int)

if not set(df["infection_label"].unique()).issubset({0, 1}):
    raise ValueError("infection_label must contain only 0 and 1.")

print("Required columns exist.")
print("Unified table shape after validation:", df.shape)

print("\nLabel distribution:")
print(df["infection_label"].value_counts())

Required columns exist.
Unified table shape after validation: (275, 26)

Label distribution:
infection_label
0    145
1    130
Name: count, dtype: int64


### Define factual profile focus types

In [5]:
LOW_PROFILE_FOCUS_TYPES = [
    {
        "profile_focus": "admission_overview",
        "secondary_focus": "procedures_and_support"
    },
    {
        "profile_focus": "procedures_and_support",
        "secondary_focus": "laboratory_values"
    },
    {
        "profile_focus": "labs_and_treatments",
        "secondary_focus": "admission_context"
    },
    {
        "profile_focus": "treatment_and_course_context",
        "secondary_focus": "procedures_and_support"
    }
]

HIGH_PROFILE_FOCUS_TYPES = [
    {
        "profile_focus": "admission_overview",
        "secondary_focus": "procedures_and_support"
    },
    {
        "profile_focus": "procedures_and_support",
        "secondary_focus": "laboratory_values"
    },
    {
        "profile_focus": "labs_and_treatments",
        "secondary_focus": "procedures_and_support"
    },
    {
        "profile_focus": "treatment_and_course_context",
        "secondary_focus": "procedures_and_support"
    }
]

print("Low focus types:", LOW_PROFILE_FOCUS_TYPES)
print("High focus types:", HIGH_PROFILE_FOCUS_TYPES)

Low focus types: [{'profile_focus': 'admission_overview', 'secondary_focus': 'procedures_and_support'}, {'profile_focus': 'procedures_and_support', 'secondary_focus': 'laboratory_values'}, {'profile_focus': 'labs_and_treatments', 'secondary_focus': 'admission_context'}, {'profile_focus': 'treatment_and_course_context', 'secondary_focus': 'procedures_and_support'}]
High focus types: [{'profile_focus': 'admission_overview', 'secondary_focus': 'procedures_and_support'}, {'profile_focus': 'procedures_and_support', 'secondary_focus': 'laboratory_values'}, {'profile_focus': 'labs_and_treatments', 'secondary_focus': 'procedures_and_support'}, {'profile_focus': 'treatment_and_course_context', 'secondary_focus': 'procedures_and_support'}]


### Built factual profile function

In [6]:
def build_factual_profile(row, profile_id, profile_version, profile_focus, secondary_focus):
    """Create a structured factual profile including all clinical vitals and lab metrics."""

    subject_id = str(row["subject_id"])
    hadm_id = str(row["hadm_id"])
    infection_label = int(row["infection_label"])

    label = map_label(infection_label)
    infection_source = map_infection_source(row)

    # Demographics and Context
    age = get_first_existing(row, ["age"])
    gender = get_first_existing(row, ["gender"])
    admission_type = get_first_existing(row, ["admission_type"])
    admission_location = get_first_existing(row, ["admission_location"])
    los_days = get_first_existing(row, ["los_days"])

    # Vitals (The columns we added to unified_table.csv)
    temp_max = get_first_existing(row, ["Temp_max"])
    hr_max = get_first_existing(row, ["HeartRate_max"])
    rr_max = get_first_existing(row, ["RespRate_max"])

    # Lab metrics
    wbc = get_first_existing(row, ["WBC_max"])
    crp = get_first_existing(row, ["CRP_max"])
    bands = get_first_existing(row, ["Bands_max"])
    neutro = get_first_existing(row, ["Neutrophils_pct_max"])

    # Others
    procedures = get_first_existing(row, ["procedures"])
    antibiotics_given = get_first_existing(row, ["antibiotics_given"])
    specimen_types = get_first_existing(row, ["specimen_types"])
    organisms = get_first_existing(row, ["organisms"])
    hospital_onset = get_first_existing(row, ["any_hospital_onset_culture"])
    expire_flag = get_first_existing(row, ["hospital_expire_flag"])

    profile = {
        "profile_id": str(profile_id),
        "source_subject_id": subject_id,
        "source_hadm_id": hadm_id,
        "profile_version": int(profile_version),
        "profile_focus": profile_focus,
        "secondary_focus": secondary_focus,

        "label": label,
        "infection_source": infection_source,
        "source_infection_label": infection_label,

        "demographics": {
            "age": to_int_or_none(age),
            "gender": clean_text_value(gender)
        },

        "admission_context": {
            "admission_type": clean_administrative_value(admission_type),
            "admission_location": clean_administrative_value(admission_location),
            "los_days": to_float_or_none(los_days)
        },

        "clinical_metrics": {
            "temp_max": to_float_or_none(temp_max),
            "heart_rate_max": to_float_or_none(hr_max),
            "resp_rate_max": to_float_or_none(rr_max)
        },

        "laboratory_values": {
            "wbc_max": to_float_or_none(wbc),
            "crp_max": to_float_or_none(crp),
            "bands_max": to_float_or_none(bands),
            "neutrophils_pct_max": to_float_or_none(neutro)
        },

        "procedures_and_support": {
            "procedures": parse_semicolon_list(procedures)
        },

        "treatments": {
            "antibiotics_given": parse_semicolon_list(antibiotics_given, clean_function=clean_drug_name)
        },

        "microbiology_context": {
            "specimen_types": parse_semicolon_list(specimen_types),
            "organisms": parse_semicolon_list(organisms),
            "any_hospital_onset_culture": normalize_bool(hospital_onset)
        },

        "outcome_context": {
            "hospital_expire_flag": normalize_bool(expire_flag)
        }
    }

    return profile

### Generate factual profiles

In [7]:
stage_a_profiles = []
profile_counter = 1

for _, row in df.reset_index(drop=True).iterrows():
    infection_label = int(row["infection_label"])

    if infection_label == 1:
        focus_types = HIGH_PROFILE_FOCUS_TYPES
    else:
        focus_types = LOW_PROFILE_FOCUS_TYPES

    for profile_version, focus_config in enumerate(focus_types, start=1):
        profile = build_factual_profile(
            row=row,
            profile_id=profile_counter,
            profile_version=profile_version,
            profile_focus=focus_config["profile_focus"],
            secondary_focus=focus_config["secondary_focus"]
        )

        stage_a_profiles.append(profile)
        profile_counter += 1

stage_a_profiles_df = pd.DataFrame([
    {
        "profile_id": profile["profile_id"],
        "source_subject_id": profile["source_subject_id"],
        "source_hadm_id": profile["source_hadm_id"],
        "profile_version": profile["profile_version"],
        "profile_focus": profile["profile_focus"],
        "secondary_focus": profile["secondary_focus"],
        "label": profile["label"],
        "infection_source": profile["infection_source"],
        "source_infection_label": profile["source_infection_label"]
    }
    for profile in stage_a_profiles
])

print("Original admissions:", len(df))
print("Stage A factual profiles:", len(stage_a_profiles))

print("\nStage A label distribution:")
print(stage_a_profiles_df["label"].value_counts())

print("\nStage A focus distribution:")
print(stage_a_profiles_df["profile_focus"].value_counts())

print("\nStage A secondary focus distribution:")
print(stage_a_profiles_df["secondary_focus"].value_counts())

display(stage_a_profiles_df.head())

Original admissions: 275
Stage A factual profiles: 1100

Stage A label distribution:


label
Low Infection Concern     580
High Infection Concern    520
Name: count, dtype: int64

Stage A focus distribution:
profile_focus
admission_overview              275
procedures_and_support          275
labs_and_treatments             275
treatment_and_course_context    275
Name: count, dtype: int64

Stage A secondary focus distribution:
secondary_focus
procedures_and_support    680
laboratory_values         275
admission_context         145
Name: count, dtype: int64


,profile_id,source_subject_id,source_hadm_id,profile_version,profile_focus,secondary_focus,label,infection_source,source_infection_label
0,1,10004235,24181354,1,admission_overview,procedures_and_support,High Infection Concern,Other/Mixed,1
1,2,10004235,24181354,2,procedures_and_support,laboratory_values,High Infection Concern,Other/Mixed,1
2,3,10004235,24181354,3,labs_and_treatments,procedures_and_support,High Infection Concern,Other/Mixed,1
3,4,10004235,24181354,4,treatment_and_course_context,procedures_and_support,High Infection Concern,Other/Mixed,1
4,5,10009628,25926192,1,admission_overview,procedures_and_support,Low Infection Concern,none,0


### Inspect profiles

In [8]:
for profile in stage_a_profiles[:3]:
    print("=" * 80)
    print("profile_id:", profile["profile_id"])
    print("label:", profile["label"])
    print("profile_focus:", profile["profile_focus"])
    print("secondary_focus:", profile["secondary_focus"])
    print()
    print(json.dumps(profile, ensure_ascii=False, indent=2))

profile_id: 1
label: High Infection Concern
profile_focus: admission_overview
secondary_focus: procedures_and_support

{
  "profile_id": "1",
  "source_subject_id": "10004235",
  "source_hadm_id": "24181354",
  "profile_version": 1,
  "profile_focus": "admission_overview",
  "secondary_focus": "procedures_and_support",
  "label": "High Infection Concern",
  "infection_source": "Other/Mixed",
  "source_infection_label": 1,
  "demographics": {
    "age": 47,
    "gender": "M"
  },
  "admission_context": {
    "admission_type": "urgent admission",
    "admission_location": "transferred from another hospital",
    "los_days": 8.98
  },
  "clinical_metrics": {
    "temp_max": 38.5,
    "heart_rate_max": 144.0,
    "resp_rate_max": 48.0
  },
  "laboratory_values": {
    "wbc_max": 21.6,
    "crp_max": null,
    "bands_max": 4.0,
    "neutrophils_pct_max": 86.0
  },
  "procedures_and_support": {
    "procedures": [
      "Arterial catheterization",
      "Endoscopic insertion of stent (tube) 

Stage B: validate the data-level fields that can actually be missing or invalid (age, sex, length of stay), drop any profile that fails, and save the clean checkpoint.
Structural fields (ids, label consistency, per-admission counts) are guaranteed by construction in Stage A, so they are not re-checked here. 
The other clinical fields (labs, procedures, antibiotics, microbiology) are legitimately sparse in MIMIC, so a missing value there is expected and simply omitted later, not an error.

In [9]:
STAGE_B_CLEAN_JSONL_PATH = PROFILES_DIR / "clean_factual_profiles.jsonl"

EXPECTED_GENDERS = {"M", "F"}


def validate_factual_profile(profile):
    """Check the always-expected admission fields that come from MIMIC."""
    issues = []

    age = profile.get("demographics", {}).get("age")
    if age is None:
        issues.append("Missing age.")
    elif not isinstance(age, int):
        issues.append("Age is not an integer.")
    elif age < 0 or age > 120:
        issues.append("Age is outside the expected range.")

    gender = profile.get("demographics", {}).get("gender")
    if gender is None:
        issues.append("Missing gender.")
    elif gender not in EXPECTED_GENDERS:
        issues.append(f"Unexpected gender value: {gender!r}.")

    los_days = profile.get("admission_context", {}).get("los_days")
    if los_days is not None:
        try:
            if float(los_days) < 0:
                issues.append("los_days is negative.")
        except (TypeError, ValueError):
            issues.append("los_days is not numeric.")

    return issues


stage_b_results = []
stage_b_clean_profiles = []

for profile in stage_a_profiles:
    issues = validate_factual_profile(profile)
    if not issues:
        stage_b_clean_profiles.append(profile)
    stage_b_results.append({
        "profile_id": profile["profile_id"],
        "label": profile["label"],
        "valid": len(issues) == 0,
        "issues": "; ".join(issues),
    })

stage_b_issues_df = pd.DataFrame(stage_b_results)

# Save only the clean factual checkpoint (no separate report files).
with STAGE_B_CLEAN_JSONL_PATH.open("w", encoding="utf-8") as file:
    for profile in stage_b_clean_profiles:
        file.write(json.dumps(profile, ensure_ascii=False) + "\n")

print("Input profiles:", len(stage_a_profiles))
print("Clean profiles:", len(stage_b_clean_profiles))
print("Removed profiles:", len(stage_a_profiles) - len(stage_b_clean_profiles))
print("Saved clean checkpoint to:", STAGE_B_CLEAN_JSONL_PATH)

if stage_b_issues_df["valid"].all():
    print("All profiles passed the data-sanity check.")
else:
    print("\nProfiles with issues:")
    display(stage_b_issues_df[stage_b_issues_df["valid"] == False].head(20))

Input profiles: 1100
Clean profiles: 1100
Removed profiles: 0
Saved clean checkpoint to: C:\Users\talme\Documents\LLM PROJECT - Improved\data\01_profiles\clean_factual_profiles.jsonl
All profiles passed the data-sanity check.


## 3 · Symptom dictionary & clinical configuration

`SYMPTOM_DICTIONARY` holds lay, non-diagnostic symptom cues keyed by infection category.
The category keys match the values in the table's `infection_categories` column so an
infection admission can be mapped to its true symptom pools. Crucially the symptom
*strings themselves* never name a diagnosis, organism, or the label — only what a
patient would feel or a nurse would observe.

In [ ]:
# Lay symptom cues (NOT diagnoses) grouped by infection category.
SYMPTOM_DICTIONARY = {
    "general_infection": [
        "fever", "chills", "malaise", "fatigue", "sweating",
        "rapid heart rate", "loss of appetite",
    ],
    "Respiratory": [
        "cough", "sputum production", "shortness of breath",
        "rapid breathing", "chest pain", "rigors",
    ],
    "Urinary": [
        "painful or burning urination", "increased bladder sensation",
        "urinary urgency", "urinary frequency",
        "lower abdominal or suprapubic pain", "suprapubic tenderness",
        "flank pain", "costovertebral angle tenderness",
        "rigors", "altered mental status", "lethargy",
    ],
    "Bloodstream/Sepsis": [
        "redness around the catheter insertion site",
        "pain or soreness around the catheter insertion site",
        "swelling around the catheter insertion site",
        "discharge from the catheter insertion site",
        "rapid heart rate", "low blood pressure",
        "rapid breathing", "altered mental status", "extreme weakness",
    ],
    "Skin/Soft tissue": [
        "redness", "swelling", "warmth", "pain", "tenderness",
        "skin discoloration", "fluid-filled blisters", "dimpled or pitted skin",
    ],
    "Other/Mixed": [
        "warmth at the incision", "cloudy or purulent wound drainage",
        "foul odor from the incision", "wound opening",
        "watery diarrhea", "mucus or blood in the stool",
        "abdominal pain or cramping", "abdominal distension",
        "nausea", "vomiting",
    ],
}
GENERAL_SYMPTOM_KEY = "general_infection"
# Every category except the non-specific "general" pool; used as the source of
# primary/noise symptoms during sampling.
SPECIFIC_CATEGORY_KEYS = [k for k in SYMPTOM_DICTIONARY if k != GENERAL_SYMPTOM_KEY]

# --- Symptom augmentation settings -----------------------------------------
AUGMENTATIONS_PER_ADMISSION = 3      # symptom-enriched profiles generated per factual profile

# Severity controls how many symptoms a profile receives. Both labels can take any
# level, but the weighting differs: Low profiles lean toward "no", High profiles lean
# toward "moderate", and they overlap most at "mild". This keeps a shared middle ground
# so the classifier cannot use severity alone as a perfect shortcut for the label.
SEVERITY_DISTRIBUTION = {
    "Low Infection Concern": {"no": 0.50, "mild": 0.35, "moderate": 0.15},
    "High Infection Concern": {"no": 0.15, "mild": 0.40, "moderate": 0.45},
}
SEVERITY_SYMPTOM_COUNT = {           # number of symptoms drawn per severity level
    "mild": (1, 2),
    "moderate": (2, 4),
}

# Symptom <-> true-category correlation knobs. Primary symptoms are drawn from the
# profile's TRUE infection categories so the picture stays clinically coherent; a
# single cross-category symptom may be added as BACKGROUND NOISE only.
P_SPECIFIC_HIGH = 0.65     # chance a High profile pulls symptoms from its true categories
P_SPECIFIC_LOW = 0.30      # chance a Low profile pulls a single non-specific complaint
P_CROSS_CATEGORY = 0.25    # chance to add one background-noise symptom from another category
MAX_SPECIFIC_SYMPTOMS = 2  # cap on primary specific picks per draw (never take a whole category)

In [11]:
# Diagnostic / label-leakage terms that must never surface in a note. Matched as
# whole words (case-insensitive) and used both to sanitize inputs and to audit output.
DIAGNOSIS_LEAKAGE_TERMS = [
    "infection", "infections", "infected",
    "sepsis", "septic", "septicemia", "septicaemia", "bacteremia", "bacteraemia",
    "pneumonia", "uti", "urinary tract infection",
    "cellulitis", "abscess", "meningitis", "endocarditis",
    "osteomyelitis", "empyema", "bronchitis", "pyelonephritis",
]
_leakage_pattern = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in DIAGNOSIS_LEAKAGE_TERMS) + r")\b",
    flags=re.IGNORECASE,
)

print("Symptom categories:", list(SYMPTOM_DICTIONARY))
print("Notes per admission:", AUGMENTATIONS_PER_ADMISSION)

Symptom categories: ['general_infection', 'Respiratory', 'Urinary', 'Bloodstream/Sepsis', 'Skin/Soft tissue', 'Other/Mixed']
Notes per admission: 3


## 4 · Dynamic symptom sampling

`sample_symptoms(label, infection_source)`:

* First samples a **severity** (`no` / `mild` / `moderate`) from `SEVERITY_DISTRIBUTION`;
  the severity sets how many symptoms are drawn (`SEVERITY_SYMPTOM_COUNT`).
* **Primary symptoms** are pulled from the profile's TRUE `infection_source` categories so
  the picture is clinically coherent; a single cross-category symptom may be added as
  background noise, and the rest is filled with non-specific `general_infection` cues.
* **Low profiles** (`infection_source == 'none'`) usually get only a non-specific complaint
  or nothing, so the model learns to separate baseline discomfort from a real signature.

In [12]:
# Symptom sampling functions (programmatic, no LLM).

def sample_severity(label):
    """Sample a symptom severity level given the profile label."""
    distribution = SEVERITY_DISTRIBUTION[label]
    levels = list(distribution.keys())
    weights = list(distribution.values())
    return rng.choices(levels, weights=weights, k=1)[0]


def map_source_to_categories(infection_source):
    """Map a profile's infection_source string to its true symptom-dictionary keys.

    Returns an empty list for Low profiles (infection_source == 'none').
    """
    if not infection_source or str(infection_source).strip().lower() == "none":
        return []
    parts = [p.strip() for p in str(infection_source).split(";")]
    return [p for p in parts if p in SYMPTOM_DICTIONARY and p != GENERAL_SYMPTOM_KEY]


def sample_symptoms(label, infection_source):
    """Pick symptoms for a profile, keeping the picture clinically coherent.

    Severity (sampled here) controls the count. Primary symptoms are drawn from the
    profile's TRUE categories so they match the underlying condition; a single
    cross-category symptom may be appended as background noise, but never as the
    main content and never without a matching primary present. General (non-specific)
    symptoms fill the rest. Returns a (severity, symptoms) tuple.
    """
    severity = sample_severity(label)
    if severity == "no":
        return severity, []

    low, high = SEVERITY_SYMPTOM_COUNT[severity]
    count = rng.randint(low, high)

    matched = map_source_to_categories(infection_source)
    is_high = label == "High Infection Concern"

    # 1) Primary symptoms drawn from the profile's TRUE categories (mostly High).
    main_specific = []
    p_specific = P_SPECIFIC_HIGH if is_high else P_SPECIFIC_LOW
    if matched and rng.random() < p_specific:
        cat = rng.choice(matched)
        bank = SYMPTOM_DICTIONARY[cat]
        k = min(len(bank), rng.randint(1, MAX_SPECIFIC_SYMPTOMS))
        main_specific = rng.sample(bank, k)
    elif not matched and rng.random() < p_specific:
        # Low has no underlying condition, so a single non-specific complaint
        # from any category is realistic and is not a clinical mismatch.
        cat = rng.choice(SPECIFIC_CATEGORY_KEYS)
        main_specific = rng.sample(SYMPTOM_DICTIONARY[cat], 1)

    # 2) Background noise: one symptom from a NON-matching category, only as a
    #    minority add-on and only when a matching primary already anchors the note.
    noise = []
    if matched and main_specific and rng.random() < P_CROSS_CATEGORY:
        noise_candidates = [k for k in SPECIFIC_CATEGORY_KEYS if k not in matched]
        if noise_candidates:
            cat = rng.choice(noise_candidates)
            noise = rng.sample(SYMPTOM_DICTIONARY[cat], 1)

    # 3) Assemble with priority: matched primaries first (never dropped for noise),
    #    then the optional noise symptom if the count leaves room, then fill the
    #    remainder with non-specific general symptoms.
    selected = list(dict.fromkeys(main_specific))[:count]
    for n in noise:
        if len(selected) < count and n not in selected:
            selected.append(n)

    general = list(SYMPTOM_DICTIONARY[GENERAL_SYMPTOM_KEY])
    rng.shuffle(general)
    for symptom in general:
        if len(selected) >= count:
            break
        if symptom not in selected:
            selected.append(symptom)

    return severity, sorted(selected)


# Quick sanity check of the sampling behavior. infection_source values must match
# the SYMPTOM_DICTIONARY keys (i.e. the simplified infection_categories).
print("Example Low severities:", [sample_severity("Low Infection Concern") for _ in range(8)])
print("Example High severities:", [sample_severity("High Infection Concern") for _ in range(8)])
print("High / Respiratory ->", sample_symptoms("High Infection Concern", "Respiratory"))
print("High / Urinary; Bloodstream/Sepsis ->", sample_symptoms("High Infection Concern", "Urinary; Bloodstream/Sepsis"))
print("Low / none ->", sample_symptoms("Low Infection Concern", "none"))

Example Low severities: ['mild', 'no', 'no', 'no', 'mild', 'mild', 'moderate', 'no']
Example High severities: ['mild', 'no', 'mild', 'mild', 'no', 'mild', 'moderate', 'mild']
High / Respiratory -> ('mild', ['loss of appetite', 'sweating'])
High / Urinary; Bloodstream/Sepsis -> ('mild', ['chills', 'low blood pressure'])
Low / none -> ('moderate', ['fatigue', 'low blood pressure', 'rapid heart rate'])


## 5 · Build symptom-enriched profiles

`build_symptom_profile` wraps each clean factual profile with a sampled severity and its
matching set of symptoms. The full factual profile is embedded inside every record, so the
downstream LLM stage has everything it needs from this single object.

In [13]:
# Build symptom-enriched profiles from clean factual profiles.

def build_symptom_profile(factual_profile, symptom_profile_id, augmentation_version):
    """Create one symptom-enriched profile wrapping a factual profile."""
    label = factual_profile["label"]
    infection_source = factual_profile["infection_source"]
    severity, symptoms = sample_symptoms(label, infection_source)

    return {
        "symptom_profile_id": str(symptom_profile_id),
        "source_profile_id": str(factual_profile["profile_id"]),
        "source_subject_id": factual_profile["source_subject_id"],
        "source_hadm_id": factual_profile["source_hadm_id"],
        "augmentation_version": int(augmentation_version),

        # Internal metadata only (not sent to the LLM in Stage E).
        "label": label,
        "infection_source": infection_source,
        "source_infection_label": factual_profile["source_infection_label"],

        # Symptom enrichment.
        "symptom_severity": severity,
        "selected_symptoms": symptoms,

        # Full factual profile is embedded so Stage E has everything it needs.
        "factual_profile": factual_profile,
    }


stage_c_symptom_profiles = []
symptom_profile_counter = 1

for factual_profile in stage_b_clean_profiles:
    for augmentation_version in range(1, AUGMENTATIONS_PER_ADMISSION + 1):
        symptom_profile = build_symptom_profile(
            factual_profile=factual_profile,
            symptom_profile_id=symptom_profile_counter,
            augmentation_version=augmentation_version,
        )
        stage_c_symptom_profiles.append(symptom_profile)
        symptom_profile_counter += 1

stage_c_profiles_df = pd.DataFrame([
    {
        "symptom_profile_id": sp["symptom_profile_id"],
        "source_profile_id": sp["source_profile_id"],
        "source_subject_id": sp["source_subject_id"],
        "source_hadm_id": sp["source_hadm_id"],
        "label": sp["label"],
        "infection_source": sp["infection_source"],
        "symptom_severity": sp["symptom_severity"],
        "num_symptoms": len(sp["selected_symptoms"]),
    }
    for sp in stage_c_symptom_profiles
])

print("Factual profiles:", len(stage_b_clean_profiles))
print("Symptom-enriched profiles:", len(stage_c_symptom_profiles))

print("\nSeverity distribution by label (proportions):")
severity_by_label = (
    pd.crosstab(stage_c_profiles_df["label"], stage_c_profiles_df["symptom_severity"], normalize="index")
    .round(3)
)
display(severity_by_label)

print("\nSeverity counts by label:")
display(pd.crosstab(stage_c_profiles_df["label"], stage_c_profiles_df["symptom_severity"]))

print("\nMean symptoms per note by label:")
display(stage_c_profiles_df.groupby("label")["num_symptoms"].mean().round(2))

Factual profiles: 1100
Symptom-enriched profiles: 3300

Severity distribution by label (proportions):


symptom_severity,mild,moderate,no
label,,,
High Infection Concern,0.406,0.442,0.152
Low Infection Concern,0.340,0.151,0.509



Severity counts by label:


symptom_severity,mild,moderate,no
label,,,
High Infection Concern,634,689,237
Low Infection Concern,592,262,886



Mean symptoms per note by label:


label
High Infection Concern    1.95
Low Infection Concern     0.95
Name: num_symptoms, dtype: float64

## 6 · Validate & save the symptom checkpoint

Each factual profile yields `AUGMENTATIONS_PER_ADMISSION` symptom-enriched variants
(resampled symptoms keep them distinct). Stage D then checks severity/symptom consistency,
scans every symptom for diagnostic leakage terms, and writes the clean
`clean_symptom_profiles.jsonl` checkpoint — the single input to the LLM stage.

In [14]:
# Inspect a couple of symptom-enriched profiles. Stage C is NOT saved separately:
# the clean, validated version is written once by Stage D, and it already embeds the
# full factual profile, so that single file is the only pre-LLM checkpoint we need.

for sp in stage_c_symptom_profiles[:2]:
    print("=" * 80)
    print("symptom_profile_id:", sp["symptom_profile_id"])
    print("label:", sp["label"])
    print("symptom_severity:", sp["symptom_severity"])
    print("selected_symptoms:", sp["selected_symptoms"])
    print("profile_focus:", sp["factual_profile"]["profile_focus"])

print("\nSeverity counts by label:")
display(pd.crosstab(stage_c_profiles_df["label"], stage_c_profiles_df["symptom_severity"]))

symptom_profile_id: 1
label: High Infection Concern
symptom_severity: moderate
selected_symptoms: ['rapid heart rate', 'sweating']
profile_focus: admission_overview
symptom_profile_id: 2
label: High Infection Concern
symptom_severity: mild
selected_symptoms: ['fever', 'vomiting']
profile_focus: admission_overview

Severity counts by label:


symptom_severity,mild,moderate,no
label,,,
High Infection Concern,634,689,237
Low Infection Concern,592,262,886


In [15]:
# Stage D: severity/symptom consistency + a leakage scan on the symptoms.
# DIAGNOSIS_LEAKAGE_TERMS / _leakage_pattern are already defined once above (in the
# symptom-configuration section), so they are reused here rather than redefined.
# Only the clean symptom profiles are saved; this single file embeds the full
# factual profile and is the input to the OpenAI/Ollama stage.

STAGE_D_CLEAN_JSONL_PATH = PROFILES_DIR / "clean_symptom_profiles.jsonl"


def find_leakage_terms(text):
    """Return any diagnostic leakage terms found in a text string."""
    if not text:
        return []
    return sorted(set(m.lower() for m in _leakage_pattern.findall(text)))


def validate_symptom_profile(sp):
    """Check severity/symptom consistency and screen symptoms for leakage terms."""
    issues = []
    severity = sp.get("symptom_severity")
    symptoms = sp.get("selected_symptoms", [])

    if severity == "no" and symptoms:
        issues.append("severity 'no' must have no symptoms.")
    if severity in {"mild", "moderate"} and not symptoms:
        issues.append(f"severity '{severity}' must have at least one symptom.")

    for symptom in symptoms:
        leaked = find_leakage_terms(str(symptom))
        if leaked:
            issues.append(f"symptom contains leakage term(s) {leaked}: '{symptom}'.")

    return issues


stage_d_results = []
stage_d_clean_symptom_profiles = []

for sp in stage_c_symptom_profiles:
    issues = validate_symptom_profile(sp)
    if not issues:
        stage_d_clean_symptom_profiles.append(sp)
    stage_d_results.append({
        "symptom_profile_id": sp["symptom_profile_id"],
        "label": sp["label"],
        "symptom_severity": sp["symptom_severity"],
        "valid": len(issues) == 0,
        "issues": "; ".join(issues),
    })

stage_d_issues_df = pd.DataFrame(stage_d_results)

# Save only the clean symptom checkpoint (the OpenAI/Ollama input).
with STAGE_D_CLEAN_JSONL_PATH.open("w", encoding="utf-8") as file:
    for sp in stage_d_clean_symptom_profiles:
        file.write(json.dumps(sp, ensure_ascii=False) + "\n")

print("Input symptom profiles:", len(stage_c_symptom_profiles))
print("Clean symptom profiles:", len(stage_d_clean_symptom_profiles))
print("Removed:", len(stage_c_symptom_profiles) - len(stage_d_clean_symptom_profiles))
print("Saved clean checkpoint to:", STAGE_D_CLEAN_JSONL_PATH)

if stage_d_issues_df["valid"].all():
    print("All symptom profiles passed the leakage / consistency check.")
else:
    print("\nProfiles with issues:")
    display(stage_d_issues_df[stage_d_issues_df["valid"] == False].head(20))

Input symptom profiles: 3300
Clean symptom profiles: 3300
Removed: 0
Saved clean checkpoint to: C:\Users\talme\Documents\LLM PROJECT - Improved\data\01_profiles\clean_symptom_profiles.jsonl
All symptom profiles passed the leakage / consistency check.
